# Manipulation d'image avec Pillow et Numpy

In [ ]:
!rm -rf sample_data
!wget http://vis-www.cs.umass.edu/lfw/lfw.tgz
!wget -O bandana.jpg https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fthumbs.dreamstime.com%2Fb%2Fman-red-bandanna-suntanned-old-jacket-wearing-sad-expression-82959941.jpg&f=1&nofb=1&ipt=64e9ac4a98500f57b5f94a224167c90d5bdadb7370ce317ca4921af3c57cdbe7&ipo=images
!tar -xf lfw.tgz
!rm lfw.tgz

In [ ]:
import pathlib
from functools import partial
from typing import Iterable, Optional, Union
from PIL import Image, ImageFilter
import numpy as np
import matplotlib.pyplot as plt

# Pillow

## Chargement d'une image

En utilisant la fonction [open](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.open) de `PIL.Image`, chargez une image de votre choix du jeu de données présent dans le dossier `lws`.

Affichez cette image avec la fonction [imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) de `matplotlib` ainsi que les informations relatives à l'image (la taille et l'encodage des couleurs)

In [ ]:
# Votre code ici

### Solution

In [ ]:
im_path = pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")
with Image.open(im_path) as im:
    print(f"L'image est de dimension {im.size}.")
    print(f"L'image est encodée en {im.mode}.")
    plt.axis("off")
    plt.imshow(im)

## Convertir en noir et blanc

À partir de votre image,
 * Transformez-la en une image en noir et blanc. Utilisez la fonction [`convert`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.convert) pour changer le [mode](https://pillow.readthedocs.io/en/latest/handbook/concepts.html#concept-modes) de l'image.
 * Sauvegarder l'image noir et blanc en png dans le **même dossier** que la photo d'origine, avec le **même nom précédé du préfixe `bw_`**. Utilisez la fonction [`save`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.save) en pensant a spécifier le format d'image `"PNG"`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
bw_im = im.convert("L")
new_path_im = im_path.with_name("bw_" + im_path.name).with_suffix(".png")
bw_im.save(new_path_im, "PNG")
plt.axis("off")
plt.imshow(bw_im, cmap="gray")

In [ ]:
!ls {new_path_im.parent}

## Découpe et collage

* À partir de l'image que vous avez choisie, découpez l'image dans un rectangle qui ressert l'image sur le visage en utilisant la fonction [`crop`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.crop).
* Créez une nouvelle image vide de même dimension que l'image d'origine avec [`Image.new`](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.new) et replacez votre image *croppé* dans celle-ci au même endroit qu'elle était placée initialement avec [`paste`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.paste).

In [ ]:
# Votre code ici

### Solution

In [ ]:
##################################################
### Fonction dédiée à l'affichage (Ignorez la) ###
##################################################
def side_by_side(
    images: Iterable[Union[Image.Image, np.ndarray]],
    title: Optional[str] = None,
    subtitles: Optional[Iterable[str]] = None,
    dpi=150,
) -> None:
    fig, axes = plt.subplots(1, len(images), dpi=dpi)
    if type(images[0]) is np.ndarray:
        mode = "gray" if len(images[0].shape) == 2 else "viridis"
    else:
        mode = "gray" if images[0].mode == "L" else "viridis"

    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap=mode)
        ax.axis("off")
        if subtitles:
            ax.set_title(subtitles[i])
    if title:
        fig.suptitle(title, y=0.85)
    fig.subplots_adjust(wspace=0, hspace=0)
    plt.show()


###################################

##########################
### Code de l'exercice ###
##########################

# Découpage du visage
face_position = (75, 10, 190, 200)
im_crop = im.crop(face_position)

# Copie du crop dans une nouvelle image de même taille
new_im = Image.new("RGB", im.size)
new_im.paste(im_crop, face_position)

side_by_side([new_im, im], "Cropped vs Original", ("Cropped", "Original"))

## Isolation des canaux de couleurs

Pillow permet d'isoler les canaux de couleurs avec la fonction [`split`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.split). Chacun des canaux isolés est stocké sous la forme d'une `Image` en mode `L` (noir et blanc) :
 * Utilisez la fonction [`split`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.split) pour séparer les canaux rgb de votre image.
 * Affichez chacun de ces canaux sous forme d'une image en noir et blanc. Pouvez vous voir des différences ?
 * Refusionnez les canaux ensemble en inversant les canaux rouge et bleu en utilisant [`merge`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.merge) et affichez le résultat.
 * Réaffichez encore une fois chacun des canaux de manière à ce que le canal rouge s'affiche en rouge et pas comme une nuance de gris. Idem pour le canal vert et bleu.

In [ ]:
# Votre code ici

### Solution

In [ ]:
(r, g, b) = im.split()
side_by_side((r, g, b), subtitles=("rouge", "vert", "bleu"))

In [ ]:
inverted_channels = Image.merge("RGB", (b, g, r))
plt.axis("off")
plt.imshow(inverted_channels)

In [ ]:
blank = Image.new("L", im.size)
rouge = Image.merge("RGB", (r, blank, blank))
vert = Image.merge("RGB", (blank, g, blank))
bleu = Image.merge("RGB", (blank, blank, b))
side_by_side((rouge, vert, bleu), subtitles=("rouge", "vert", "bleu"), dpi=300)

## Transformation géométrique


Effectuez et affichez les transformations suivantes à partir de l'image d'origine :
 * Effectuez une rotation d'un quart de tour dans le sens horaire (voir [`rotate`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.rotate))
 * Inversez l'image comme un miroir (voir [`transpose`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.transpose))
 * Redimensionnez l'image pour obtenir une image où la hauteur et largeur sont doublées (voir [`resize`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.resize))
 * Redimensionnez l'image pour obtenir une image où la hauteur et largeur sont divisées par 5

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Rotation horaire de 90°
plt.imshow(im.rotate(-90))
plt.axis("off")
plt.show()

# Image miroir (inversion gauche-droite)
plt.imshow(im.transpose(Image.Transpose.FLIP_LEFT_RIGHT))
plt.axis("off")
plt.show()

# Redimensionnnement doublant la taille de l'image
plt.imshow(im.resize((2 * x for x in im.size)))
plt.axis("off")
plt.show()

# Redimensionnnement divisant par 5 la taille de l'image
plt.imshow(im.resize((x // 5 for x in im.size)))
plt.axis("off")
plt.show()

## Filtres
Utilisez la fonction [`filter`](https://pillow.readthedocs.io/en/latest/reference/Image.html#PIL.Image.Image.filter) et appliquer les filtres suivants :
 * `BLUR`
 * `SMOOTH`
 * `FIND_EDGES`
 * `GaussianBlur`


In [ ]:
# Votre code ici

### Solution

In [ ]:
blurred = im.filter(ImageFilter.BLUR)
side_by_side((blurred, im), "Blurred VS Original")

smoothed = im.filter(ImageFilter.SMOOTH)
side_by_side((smoothed, im), "Smoothed VS Original")

edges = im.filter(ImageFilter.FIND_EDGES)
side_by_side((edges, im), "Edges detection")

gauss_blur = im.filter(ImageFilter.GaussianBlur())
side_by_side((gauss_blur, im), "Gaussian blur VS Original")

# Manipulation d'image avec numpy (*prérequis: numpy et matplotlib*)

Si Pillow propose de nombreuses méthodes de manipulations d'images, la librairie reste principalement sur les fonctions classiques de manipulations et sur les fonctions de sauvegarde, chargement et changement de mode. Elle manque de capacité de manipulations avancées et d'une plus grande souplesse dans les opérations proposées.

Numpy, librairie de calcul scientifique, permet d'implémenter des opérations plus complexes en considérant les images comme des tableaux multi-dimensionnels.

## Transformation d'une image en un tableau numpy

Commencons simplement par transformer une image en tableau numpy. Numpy et Pillow sont interopérable et permettent de passer d'un tableau a une image (et vice-versa) en une ligne de code.

 * Chargez l'image de votre choix avec pillow comme vu précédemment, puis créez directement un `numpy.array` à partir de celle-ci
 * Affichez les caractéristiques du tableau : dimensions et type de données stockées
 * Affichez ensuite ce tableau numpy avec `imshow`. Vous ne devriez pas voir de différence par rapport à un affichage de l'image pillow

In [ ]:
# Votre code ici

### Solution

In [ ]:
with Image.open(im_path) as image_pillow:
    im_np = np.array(image_pillow)
    print(
        f"Le tableau numpy est de dimension {im_np.shape}.\
  La dernière dimension correspond au nombre de layers ({im_np.shape[-1]})."
    )
    print(f"Les données stockées dans le tableau sont des {im_np.dtype}.")

plt.axis("off")
plt.imshow(im_np)
plt.show()

## Séparation des canaux de couleurs

Comme pour l'exercice de séparation des couleurs avec Pillow, effectuez l'affichage de chaque canal, en utilisant seulement le tableau numpy contenant votre image.

In [ ]:
# Votre code ici

### Solution

In [ ]:
side_by_side(
    (
        im_np[:, :, 0],
        im_np[:, :, 1],
        im_np[:, :, 2],
    ),  # Selection indépendante des canaux
    subtitles=("rouge", "vert", "bleu"),
)

## Histogramme d'une image

Un outil classique en traitement d'image est l'[histogramme des valeurs de pixels](https://fr.wikipedia.org/wiki/Histogramme_(imagerie_num%C3%A9rique)).

Réalisez les exercices suivants :

1. Convertissez votre image en noir et blanc et récupérez le tableau numpy associé. Vous allez produire l'histogramme de votre image et l'afficher :
  * L'axe des $x$ correspond à l'ensemble des valeurs possibles pour un pixel (0 à 255)
  * L'axe des $y$ correspond à la fréquence d'appartion de chaque valeur de pixel dans votre image

  Vous pouvez réalisez cet exercice de deux manières : soit en calculant à la main les fréquences de chaque valeur et produire un [`plt.bar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html), soit d'utiliser directement [`plt.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html).
2. À partir de l'histogramme noir et blanc, plottez la ligne représentant la somme cumulée des fréquences des valeurs successives de pixel.
3. Produisez l'histogramme de l'image en couleur en faisant un histogramme par canal.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def plot_hist(im: np.ndarray) -> np.ndarray:
    im_hist, _ = np.histogram(im, bins=256)
    # On multiplie par le ratio entre la valeur maximale de l'histogramme et
    # la somme de toutes ses valeurs pour que la somme cumulée n'écrase pas
    # l'histogramme à cause du rapport de taille
    im_cumsum_hist = np.cumsum(im_hist) * np.max(im_hist) / np.sum(im_hist)
    plt.hist(im.flatten(), bins=256, range=(0, 255))
    plt.plot(np.arange(256), im_cumsum_hist)
    plt.show()
    return im_hist


im_bw_np = np.array(im.convert("L"))
im_hist = plot_hist(im_bw_np)

Différentes représentations de l'histogramme d'une image RGB

In [ ]:
plt.hist(im_np[:, :, 0].flatten(), bins=256, range=(0, 255), color="r", alpha=0.5)
plt.hist(im_np[:, :, 1].flatten(), bins=256, range=(0, 255), color="g", alpha=0.5)
plt.hist(im_np[:, :, 2].flatten(), bins=256, range=(0, 255), color="b", alpha=0.5)
plt.show()

In [ ]:
import seaborn as sns

sns.kdeplot(x=im_np[:, :, 0].flatten(), label="red", color="r")
sns.kdeplot(x=im_np[:, :, 1].flatten(), label="green", color="g")
sns.kdeplot(x=im_np[:, :, 2].flatten(), label="blue", color="b")
plt.legend()

In [ ]:
# Extraire les canaux RGB
r_channel = im_np[:, :, 0]
g_channel = im_np[:, :, 1]
b_channel = im_np[:, :, 2]

# Créer des histogrammes 1D pour chaque canal
r_hist, r_bins = np.histogram(r_channel, bins=256, range=(0, 256))
g_hist, g_bins = np.histogram(g_channel, bins=256, range=(0, 256))
b_hist, b_bins = np.histogram(b_channel, bins=256, range=(0, 256))

# Créer un plot 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

# Créer les plans pour les histogrammes
x = np.arange(0, 256)
ax.bar(x, r_hist, zs=0, zdir="y", width=1, color="r", alpha=0.7)
ax.bar(x, g_hist, zs=1, zdir="y", width=1, color="g", alpha=0.7)
ax.bar(x, b_hist, zs=2, zdir="y", width=1, color="b", alpha=0.7)

# Personnaliser l'apparence
ax.set_xlabel("Valeur des canaux")
ax.set_ylabel("Canal")
ax.set_zlabel("Fréquence")
ax.set_title("Histogramme 3D des canaux RGB")

plt.show()

### Utilisation de l'histogramme pour corriger le contraste
Une utilisation simple de l'histogramme est la correction de contraste. En fonction de l'image choisie, vous pourriez avoir un histogramme dont la luminance (~ la concentration de valeurs de pixel dans un intervalle) se concentre sur les noirs, les gris ou les blancs sans utiliser complètement la gamme de couleurs possibles.

Nous allons appliquer une correction $T(k)$ sur les valeurs de pixel de notre image en **noir et blanc** en utilisant la somme cumulée de l'histogramme et définie telle que suit :

$$
T(k) = \frac{255}{n}\sum_{j=0}^k n_j
$$
avec :
 * $k$ la valeur d'un pixel (entre 0 et 255)
 * $n_j$ le nombre de pixels de valeur $j$
 * $n$ le nombre de pixels
 * $\sum_{j=0}^k n_j$ la fréquence cumulée des valeurs de pixels de 0 à $k$

Réalisez les étapes suivantes :
1. Ecrivez la fonction de transformation $T$ de prototype
```python
def T(pixel_value:int, cumsum_hist:np.ndarray) -> int :
```
2. Appliquez la fonction de transformation $T$ sur votre image en noir et blanc pour chaque pixel en appelant la fonction `egalisation` fournie ci-dessous.
3. Affichez votre image resultante et l'histogramme associé.

In [ ]:
def T(pixel_value: int, cumsum_hist: np.ndarray) -> int:
    """
    pixel_value: valeur d'un pixel en niveau de gris de 0 à 255
    cumsum_hist: liste des sommes cumulées de l'histogramme de l'image
                tel que cumsum_hist[k] correspond à la somme cumulée de
                la valeur de pixel zero, jusqu'a la k-ième
    """
    # Votre code ici
    pass


def egalisation(im: np.ndarray) -> np.ndarray:
    """
    À partir de l'image en paramètre, renvoie l'image avec égalisation de
    contraste
    """
    new_im = im.copy()
    # Calcul de la somme cumulée de l'histogramme
    cumsum_hist = np.cumsum(np.histogram(new_im, bins=256)[0])
    # Vectorisation de la fonction T
    T_vect = np.vectorize(partial(T, cumsum_hist=cumsum_hist))
    # Application de la fonction T sur new_im
    return T_vect(new_im)


# Transformez votre image noir et blanc
# Votre code ici

Affichage de l'image résultante et de son histogramme

In [ ]:
# Votre code ici

### Solution

In [ ]:
def T(pixel_value: int, cumsum_hist: np.ndarray) -> int:
    return int(255 / cumsum_hist[-1] * cumsum_hist[pixel_value])


def egalisation(im: np.ndarray) -> np.ndarray:
    """
    À partir de l'image en paramètre, renvoie l'image avec égalisation de
    contraste
    """
    new_im = im.copy()
    # Calcul de la somme cumulée de l'histogramme
    cumsum_hist = np.cumsum(np.histogram(new_im, bins=256)[0])
    # Vectorisation de la fonction T
    T_vect = np.vectorize(partial(T, cumsum_hist=cumsum_hist))
    # Application de la fonction T sur new_im
    return T_vect(new_im)


contrasted = egalisation(im_bw_np)
side_by_side((contrasted, im_bw_np), subtitles=("Constraste augmenté", "Original"))
plt.show()
_ = plot_hist(contrasted)

## Fusion d'image (difficulté +)

L'objectif de cet exercice est de fusionner deux images l'une dans l'autre de manière progessive le long de l'axe des abscisses.

1. Charger deux images de votre choix dans des tableaux numpy dans deux variables `im_1` et `im_2`
2. Définisser un fonction qui prend deux colonnes de pixels pour produire une nouvelle colonne qui est la combinaison des deux images dans une certaine proportion $\alpha$ tel que :
$$new\_col \leftarrow \alpha\cdot col_1 + (1-\alpha)\cdot col_2$$
avec $\alpha\in [0,1]$
3. Iterer sur les colonnes de votre image en appliquant la function de fusion et en changeant la proportion $\alpha$ de telle façon que l'image `im_1` s'affiche a 100\% à gauche et l'image `im_2` s'affiche à 100\% à gauche. Au milieu de l'image resultante, les deux images s'affichent à 50\%

In [ ]:
# Votre code ic

### Solution

In [ ]:
def merge(im1, im2):
    filter = np.zeros(im1.shape)
    for i in range(im1.shape[1]):
        filter[:, i] = i / im1.shape[1]
    merged = im1 * filter + im2 * (1 - filter)
    return merged.astype("uint8")


im_path_1 = pathlib.Path("lfw/Chris_Rock/Chris_Rock_0001.jpg")
im_1 = np.array(Image.open(im_path_1))
im_path_2 = pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")
im_2 = np.array(Image.open(im_path_2))

merged = merge(im_1, im_2)
plt.imshow(merged)

## Isolation de couleurs (difficulté ++)
Mettons à présent toutes vos compétences en pratique en utilisant Pillow et Numpy.

L'objectif de cet exercice est de produire, à partir d'une image en couleur, un image en noir et blanc dans laquelle une couleur a été conservée (voir ci-dessous).

![](https://qph.cf2.quoracdn.net/main-qimg-605444170d2024a8ffe656a57c6c766d-lq)

Vous trouvez à la racine de votre dossier un fichier `bandana.jpg` correspondant à l'image ci-dessous :
![bandana](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fthumbs.dreamstime.com%2Fb%2Fman-red-bandanna-suntanned-old-jacket-wearing-sad-expression-82959941.jpg&f=1&nofb=1&ipt=64e9ac4a98500f57b5f94a224167c90d5bdadb7370ce317ca4921af3c57cdbe7&ipo=images)

L'objectif de cet exercice est de produire une image en noir et blanc qui conserve le rouge du bandana.

Voici les informations dont vous aurez besoin pour réaliser cette exercice :
 * Utilisez au mieux toutes les techniques vues précédemment
 * Une image `RGB` ou chaque canal est identique correspond a une image en noir et blanc
 * Le format `RGB` est très mal adapté pour filtrer une couleur spécifique, il faudra lui préférer le mode [`HSV`](https://fr.wikipedia.org/wiki/Teinte_saturation_lumi%C3%A8re). Le premier canal (`H`) de ce format correspond à la couleur du pixel exprimée sur un cercle de couleur. En HSV, pillow exprime les couleurs entre 0 et 255 sur ce canal. 0 correspond au rouge et 255 aussi (quoique très subtilement plus violacé). Le cyan est donc autour de 128 par opposition (ne vous fiez pas aux valeurs indiqué sur le dessin ci-dessous).
 ![](https://upload.wikimedia.org/wikipedia/commons/5/5c/Color_wheel_with_degree.png)
 * Vous n'aurez pas besoin de travailler sur les canaux `S` et `V`
 * Rappellez vous qu'une image pillow peut être transformée en tableau array avec `np.array`. Inversement un tableau numpy peut être transformé en une image pillow avec [`Image.fromarray`](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.fromarray)

In [ ]:
# Votre code ici

### Solution

In [ ]:
def one_color(im: np.ndarray, color_range, exclude=False):
    # Convertion en HSV pour simplifier le choix de couleur
    im_hsv = np.array(Image.fromarray(im, "RGB").convert("HSV"))
    # Conversion de l'image en noir et blanc
    bw = np.array(Image.fromarray(im, "RGB").convert("L"))
    # Ajout d'une dimension pour specifier les canaux et simplifier les calculs
    # Une image noir et blanc de dimension (x, y) devient de dimension (x, y, 1)
    bw = bw.reshape(im.shape[:2] + (1,))

    # Filtre sur le canal H specifiant l'intervalle de valeurs pour choisir
    # le spectre de couleur qui ne sera pas transformé en noir et blanc
    filter = (color_range[0] < im_hsv[:, :, 0]) & (color_range[1] > im_hsv[:, :, 0])
    # Option pour choisir un intervalle de valeur de couleurs à exclure plutot
    # qu'à inclure
    if exclude:
        filter = 1 - filter

    # Ajout de la dimension de canal
    filter = filter.reshape(im.shape[:2] + (1,))
    # Filtrage
    im_1c = im * filter + bw * (1 - filter)
    return im_1c.astype("uint8")


bandana = np.array(Image.open("bandana.jpg"))
# On prend les valeurs entre 235 et 251 qui correspondent presque à toutes les
# couleurs du bandana
bandana_1c = one_color(bandana, (235, 251))
side_by_side((bandana, bandana_1c))